# Lab 07: The Periodic Quasar

**ASTR 457, Fall 2026, posted Thu Oct 22, due Wed Oct 28 by Noon (fork → PR, as always)**

*Why this lab: nearly every light curve in astronomy is correlated noise with, possibly, a signal inside it, and "I found a period" is one of the most-retracted sentences in the literature. Knowing what the noise is, and testing a claim against that noise instead of against white noise, is the difference between a discovery and a red-noise artefact. A Gaussian process is the tool that lets you say what the noise is.*

A survey team has been monitoring quasars in the g band. In one of them they report a tentative periodic modulation, and a quasar whose brightness repeats could be two supermassive black holes in orbit around each other, which would be a spectacular discovery. Every one of you has your own quasar: `data/<your netid>.csv`, with columns `t_days` (days since the first epoch), `mag_g` (g-band magnitude) and `sigma_mag` (the pipeline's per-epoch uncertainty, which is honest). It was observed in three observing seasons over about 970 days (the quasar is up for about eight months a year), 240 to 290 epochs in all.

Quasars vary all the time, with no period at all: their light curves are well described by a **damped random walk** (DRW). The next value is a decayed copy of the last one plus a random kick. It is a Gaussian process with the exponential covariance function,

$$k(\Delta t) = \sigma^2\,\exp\!\left(-\frac{|\Delta t|}{\tau}\right),$$

where $\sigma$ is the rms variability in magnitudes and $\tau$ is the damping time in days: how long the light curve remembers where it was. (Samaresh Mondal's colloquium on Tuesday was about telling a tidal disruption flare from AGN variability using how much a light curve varies at each time lag, the structure function. This lab is the cousin problem: telling a period from AGN variability.)

**I know how your light curve was made, and you don't.** Specifically, I know $\sigma$ and $\tau$, and I know whether a real periodic signal is in there. For some of you there is one. For some of you there is not, and the honest answer is "no evidence". Your classmates' light curves are different, and so is the sampling.

**How this is graded.** On whether your answers are *right* and your uncertainties are *honest*: your $\sigma$ and $\tau$ are compared to your truth in units of your own reported uncertainty. Anti-sandbagging cap: your 68% interval on $\ln\tau$ may not be wider than $3\times$ a reference width I compute for your dataset, with the same prior as the one stated in Part 2 if you quote a posterior, or from the profile likelihood if you quote a profile interval. For the periodicity question I grade the *evidence you present*, not just the verdict: for a given draw, "no evidence" can be exactly right, and "a period" can be right for the wrong reason.

**AI policy reminder.** Use whatever tools you like, including AI assistants, and document it in Part 6. You may be selected to defend this lab orally; the defense is easy if you did the work.

Weights: Part 1 10%, Part 2 30%, Part 3 20%, Part 4 25%, Part 5 5%, Part 6 10%.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from astropy.timeseries import LombScargle

NETID = 'yournetid'   # <-- set this
d = np.genfromtxt(f'data/{NETID}.csv', delimiter=',', names=True)
t, y, e = d['t_days'], d['mag_g'], d['sigma_mag']
T = t.max() - t.min()
print(f"{t.size} epochs over {T:.0f} days; median quoted error {np.median(e):.3f} mag; rms of the light curve {y.std():.3f} mag")

## Part 1: Look first, then the periodogram and its window (10%)

Plot the light curve with its error bars, and look at it. Then compute a Lomb-Scargle periodogram (`astropy.timeseries.LombScargle`, using the errors) over periods from 20 days to $T/2$, where $T$ is the length of your light curve. (Why not longer than $T/2$? Say in a sentence.) Compute the **window function**: the periodogram of the sampling alone, with the data replaced by a constant (`LombScargle(t, np.ones_like(y), fit_mean=False, center_data=False)`; scale it to the same maximum as your periodogram, so you can compare where the peaks are; the heights mean nothing). Use a frequency grid with a spacing of at most $1/(5T)$. Plot both. Then report the tallest peak and its false-alarm probability from astropy (`false_alarm_probability`).

Answer, in a few sentences: what period is the tallest peak, which other peaks are probably aliases of it or of the sampling, and what noise model does astropy's false-alarm probability assume?

In [ ]:
# Part 1: your work here

**ANSWER (a few sentences):**

*your answer goes here*

## Part 2: The Gaussian process (30%)

Model the light curve as a constant mean $\mu$ plus a DRW plus the measurement noise. The data $\mathbf{y}$ are one draw from a multivariate Gaussian,

$$\ln L = -\tfrac12 (\mathbf{y}-\mu)^T C^{-1} (\mathbf{y}-\mu) - \tfrac12 \ln|C| - \tfrac{N}{2}\ln 2\pi, \qquad C_{ij} = \sigma^2 e^{-|t_i-t_j|/\tau} + \sigma_i^2\,\delta_{ij}.$$

Here $\sigma_i$ is the *quoted* error of epoch $i$, so $\sigma$ without a subscript is the DRW amplitude. $N$ is about 250, so $C$ is a $250\times250$ matrix and one likelihood evaluation costs milliseconds with a Cholesky factorization (`scipy.linalg.cho_factor` and `cho_solve`; get $\ln|C|$ from the diagonal of the factor, never from `np.linalg.det`).

1. **Write the log-likelihood, the log-prior and the log-posterior** for $(\mu, \ln\sigma, \ln\tau)$. State every prior and justify it in one line each. $\sigma$ and $\tau$ are scale parameters: use priors that are uniform in $\ln\sigma$ and $\ln\tau$, bounded to $0.01 < \sigma < 3$ mag and $10 < \tau < 4000$ d, and say what the bound on $\tau$ does to your result.
2. **Fit it.** Find the maximum-likelihood point from more than one starting value, and say why. Then get honest uncertainties on $\sigma$ and $\tau$ with either `emcee` (you know how from Lab 05; say how you checked convergence) or a profile likelihood over $\tau$ (maximize over $\mu$ and $\sigma$ at each $\tau$ on a grid from 10 to 4000 d, and read the interval off $\Delta(-2\ln L) = 1$). If you run `emcee`, a few thousand steps is enough and takes a few minutes; do not run for ten. Report $\sigma$ and $\tau$ as medians with 68% intervals (or best fit with a profile interval), and $T/\tau$. The $\tau$ interval is probably not symmetric: report it the way it is, and say what makes $\tau$ hard to measure from a light curve of this length.
3. **Compare kernels.** Refit with a squared-exponential kernel, $k = \sigma^2 e^{-\Delta t^2/2\ell^2}$, which has the same number of parameters. Which has the higher maximum $\ln L$, and by how much? What does that say about how smooth a quasar light curve is?
4. **Predict across a gap.** Plot the posterior-mean prediction and its 68% band for your best kernel over a regular time grid that spans one of the seasonal gaps, together with the data. State whether your band is for the *latent* light curve or for a new *measurement*, and why that matters when you compare it with data points.
5. **Test the prediction.** Hold out four non-overlapping blocks of 30 consecutive days, each inside one observing season and at least 60 days apart (pick them at random with a fixed seed, and redraw until they satisfy that). Fit on the rest, predict the held-out points, and compare them with the band **that includes the measurement noise**. Report how many points you held out, the fraction inside your 68% band, and the $R^2$ of the predictions (against the mean of the training data). With about 40 held-out points the fraction has a scatter of roughly $\pm 7\%$, so judge "honest" accordingly. Do the same for the other kernel. Is the band honest?

**FINAL ANSWER:** $\sigma = $ ___ (+___ / −___) mag, $\tau = $ ___ (+___ / −___) d, $T/\tau = $ ___; the exponential kernel beats the squared exponential by $\Delta\ln L = $ ___; ___ of ___ held-out points are inside the 68% band.

In [ ]:
# Part 2: your work here

**FINAL ANSWER:** *fill in the numbers above, here, in your own words*

## Part 3: Is it periodic? (20%)

A periodic claim is a claim that *this peak is too tall to be noise*, so it is only as good as your model of the noise.

1. **State your decision rule before you run anything in this part.** Write, in a markdown cell *above* your code, the empirical $p$-value below which you would claim a period, and why you chose it. (A rule written after you have seen the answer does not count.)
2. **Build the right null.** Simulate at least 1000 DRW light curves (300 cannot resolve a $p$-value of 0.01) with your Part 2 maximum-likelihood $\mu$, $\sigma$ and $\tau$, **at your epochs and with your errors** (draw $\mathbf{y} = \mu + L\mathbf{z} + $ noise with $L$ the Cholesky factor of the DRW covariance and $\mathbf{z}$ standard normal, plus a Gaussian draw with each $\sigma_i$). Run the same Lomb-Scargle search (20 d to $T/2$) on every one, record the *tallest* peak in each, and report the fraction of simulations whose tallest peak is at least as tall as yours. That is your empirical $p$-value. Compare it with astropy's false-alarm probability from Part 1, and explain the difference.
3. **Ask the GP what a periodic signal would add.** Fit $y = \mu + A\sin(2\pi t/P + \phi) + \mathrm{DRW}$. For *fixed* $\sigma$ and $\tau$ (the Part 2 maximum-likelihood values; say how your $p$-value changes at the two ends of your $\tau$ interval) and a fixed period, the amplitudes enter linearly, so the best $A\cos\phi$ and $A\sin\phi$ come from generalized least squares: $\hat\theta = (X^T C^{-1} X)^{-1} X^T C^{-1}\mathbf{y}$ with columns $1, \cos 2\pi t/P, \sin 2\pi t/P$. Scan period from 20 d to $T/2$, record the best $\Delta\chi^2$ against the constant-mean model, and compare it to the same statistic on your simulated null light curves. (The simulations are what account for the look-elsewhere effect: you searched many periods.)
4. **Decide.** Is there evidence for a periodic signal in your light curve, by your own rule? If you claim one, give a period and an amplitude with their uncertainties. If you do not, say what you would need to see. One sentence: should the team announce a binary candidate on this evidence?

In [ ]:
# Part 3: your work here

**ANSWER (the decision rule first, then the verdict):**

*your answer goes here*

## Part 4: The referee report (25%)

The file `ai_solution.ipynb` in this directory is a complete, tidy, confident analysis of this exact problem, produced by an AI assistant. It runs top to bottom without errors, the plots are handsome, and the prose is fluent. It is also wrong, in at least **three** distinct, substantive ways (cosmetic nitpicks and style don't count; "it used a 5-point cross-validation and I used four blocks" is not a finding by itself).

Write a referee report:

1. **Identify** at least three substantive statistical errors.
2. **Demonstrate** each one quantitatively on *your* dataset: show, with a number or a plot, what the error does to the inference. A demonstration can come out negative: if an error turns out not to move *your* numbers, showing that, and saying what it would take for it to bite, is a valid finding.
3. **State** the correct treatment (you built most of it in Parts 2 and 3).

Fourth time you have done this. You should be finding these in minutes now.

In [ ]:
# Part 4: your work here

**REFEREE REPORT:**

*your report goes here*

## Part 5: A live light curve (5%)

Everything so far was a light curve I simulated. This one is real, and it is recent. Look up your quasar in `data/live_objects.csv` (columns `netid`, `ztf_oid`). The function below pulls its g-band light curve from the public ALeRCE API for the Zwicky Transient Facility (ZTF). If the API is down, it falls back to `data/live/<oid>.csv`, a copy I saved on Oct 21.

In [ ]:
import requests, os
from astropy.time import Time

def get_ztf_g(oid):
    """g-band (mjd, mag, err) for a ZTF object from the ALeRCE API (corrected magnitudes, dubious points removed).
    Falls back to the cached copy data/live/<oid>.csv, and says so."""
    try:
        r = requests.get(f'https://api.alerce.online/ztf/v1/objects/{oid}/lightcurve', timeout=60)
        r.raise_for_status()
        rows = [(q['mjd'], q['magpsf_corr'], q['sigmapsf_corr_ext']) for q in r.json()['detections']
                if q['fid'] == 1 and q.get('corrected') and not q.get('dubious')
                and q.get('magpsf_corr') is not None and q.get('sigmapsf_corr_ext') is not None]
        a = np.array(sorted(rows))
        print(f'{oid}: {a.shape[0]} g-band detections from the live API')
    except Exception as ex:
        print(f'API failed ({ex!r}); using the cached copy')
        a = np.genfromtxt(f'data/live/{oid}.csv', delimiter=',', skip_header=2)
    return a[:, 0], a[:, 1], a[:, 2]

live = np.genfromtxt('data/live_objects.csv', delimiter=',', names=True, dtype=None, encoding=None)
OID = [r['ztf_oid'] for r in live if r['netid'] == NETID][0]
mjd, mag, merr = get_ztf_g(OID)
print(f"{OID}: MJD {mjd.min():.1f} to {mjd.max():.1f} ({(mjd.max() - mjd.min()) / 365.25:.1f} yr); the last detection is MJD {mjd.max():.1f}; today is MJD {Time.now().mjd:.0f}")

Apply your own standard from Part 3 to this light curve, **state before you run it** what result would make you call it periodic, and then run it (nightly means are fine if the full sample is too slow for the simulations). Two or three sentences: is there evidence for a period, what is different about this sampling from your simulated light curve (look at the last detection date above, and at how the cadence differs), and what would you do next with a week more of time?

In [ ]:
# Part 5: your work here

**ANSWER (the criterion first, then the result):**

*your answer goes here*

## Part 6: AI-use appendix and verification plan (10%)

1. **AI use**: which tools did you use, for what, what did they get wrong, and how did you catch it? Honesty is graded; "I didn't use any" is fine if true.
2. **Reflection**: what did you use AI for in this lab, and what would have gone wrong in your answer if you had not checked its output? Name the specific thing, not "it could have made a mistake".
3. **Verification plan**: the checks you ran before trusting your Part 2 and Part 3 numbers, and what failure would have looked like for each. One you should seriously consider: generate a synthetic light curve with parameters *you* choose (a DRW with a known $\sigma$ and $\tau$, at *your* epochs and errors, with and without a sinusoid you plant), run your whole pipeline on it, and see whether the 68% intervals contain what you put in and whether your Part 3 test finds the sinusoid you planted and stays quiet when you did not plant one. One realisation is a spot check, not a calibration: say how many you would need.

**AI-USE APPENDIX:**

*your answer goes here*

**REFLECTION:**

*your answer goes here*

**VERIFICATION PLAN:**

*your plan goes here*